# 🥈 SILVER LAYER — Cleaning, Imputasi, Normalisasi

**Tujuan:** Filter 4 indikator SDG 5, buang kawasan agregat, reshape data, imputasi missing value, normalisasi Min-Max.

| Input | Output |
|-------|--------|
| `/data/bronze/main_data.parquet` | `/data/silver/sdg5_normalized.parquet` |
| `/data/bronze/country_meta.parquet` | 217 baris × 8 kolom, zero missing |

**4 Indikator SDG 5:**
| Kode | Nama | Coverage |
|------|------|----------|
| SP.ADO.TFRT | Adolescent Fertility Rate | 100% |
| SH.STA.MMRT | Maternal Mortality Ratio | 89.4% |
| SG.GEN.PARL.ZS | Women in Parliament (%) | 87.6% |
| SL.TLF.ACTI.FE.ZS | Female Labor Force Participation (%) | 86.1% |

In [5]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, expr, avg, first, broadcast

# Inisialisasi SparkSession (local mode)
spark = SparkSession.builder \
    .appName("SDG5-Gender-Analysis") \
    .master("local[*]") \
    .config("spark.driver.memory", "2g") \
    .config("spark.sql.shuffle.partitions", "8") \
    .config("mapreduce.fileoutputcommitter.marksuccessfuljobs", "false") \
    .config("mapreduce.fileoutputcommitter.algorithm.version", "2") \
    .getOrCreate()

# Load data dari output Bronze Layer
df_main = spark.read.parquet("/data/bronze/main_data.parquet")
df_country = spark.read.parquet("/data/bronze/country_meta.parquet")
print("✅ Data Bronze berhasil dimuat.")

✅ Data Bronze berhasil dimuat.


In [6]:
# ============================================================
# STEP 1 — Filter 4 Indikator SDG 5
# ============================================================
TARGET_INDICATORS = [
    "SP.ADO.TFRT",       # Adolescent Fertility Rate
    "SH.STA.MMRT",       # Maternal Mortality Ratio
    "SG.GEN.PARL.ZS",    # Women in Parliament
    "SL.TLF.ACTI.FE.ZS"  # Female Labor Force Participation
]

df_filtered = df_main.filter(col("Indicator Code").isin(TARGET_INDICATORS))
print(f"Setelah filter indikator: {df_filtered.count():,} baris")

# ============================================================
# STEP 2 — Filter Negara Real (Buang Kawasan Agregat)
# ============================================================
# Gunakan broadcast join (jauh lebih efisien daripada .collect())
real_countries_df = df_country.filter(
    col("Region").isNotNull()
).select("Country Code")

print(f"Total negara real (punya Region): {real_countries_df.count()}")

df_filtered = df_filtered.join(broadcast(real_countries_df), on="Country Code", how="inner")
print(f"Setelah filter negara: {df_filtered.count():,} baris")

Setelah filter indikator: 1,060 baris
Total negara real (punya Region): 216
Setelah filter negara: 848 baris


In [7]:
# ============================================================
# STEP 3 — Pivot Wide ke Long (Melt Kolom Tahun 2015-2022)
# ============================================================
YEARS = [str(y) for y in range(2015, 2023)]

stack_expr = "stack({}, {}) as (year, value)".format(
    len(YEARS),
    ", ".join([f"'{y}', `{y}`" for y in YEARS])
)

df_long = df_filtered.select(
    "Country Name", "Country Code", "Indicator Code",
    expr(stack_expr)
).filter(col("value").isNotNull()) \
 .withColumn("value", col("value").cast("double"))

print(f"Setelah melt (long format): {df_long.count():,} baris")

# ============================================================
# STEP 4 — Rata-rata Per Negara Per Indikator (2015-2022)
# ============================================================
df_wide = df_long.groupBy("Country Code", "Country Name", "Indicator Code") \
    .agg(avg("value").alias("avg_value")) \
    .groupBy("Country Code", "Country Name") \
    .pivot("Indicator Code", TARGET_INDICATORS) \
    .agg(first("avg_value"))

print(f"\nSetelah pivot ke wide: {df_wide.count()} baris (negara)")
print(f"Kolom: {df_wide.columns}")

Setelah melt (long format): 6,147 baris

Setelah pivot ke wide: 212 baris (negara)
Kolom: ['Country Code', 'Country Name', 'SP.ADO.TFRT', 'SH.STA.MMRT', 'SG.GEN.PARL.ZS', 'SL.TLF.ACTI.FE.ZS']


In [8]:
# ============================================================
# STEP 5 — Join Metadata Negara (Region & Income Group)
# ============================================================
df_country_slim = df_country.select(
    "Country Code", "Short Name", "Region", "Income Group"
)

df_joined = df_wide.join(df_country_slim, on="Country Code", how="left")

print(f"Setelah join metadata: {df_joined.count()} baris")
print(f"Kolom: {df_joined.columns}")

# ============================================================
# CEK MISSING VALUE SEBELUM IMPUTASI
# ============================================================
print("\n" + "=" * 50)
print("⚠️  MISSING VALUE SEBELUM IMPUTASI")
print("=" * 50)
for col_name in TARGET_INDICATORS:
    # Gunakan backtick karena nama kolom mengandung titik
    null_count = df_joined.filter(col(f"`{col_name}`").isNull()).count()
    pct = null_count / 217 * 100
    print(f"  {col_name}: {null_count} missing ({pct:.1f}%)")

Setelah join metadata: 212 baris
Kolom: ['Country Code', 'Country Name', 'SP.ADO.TFRT', 'SH.STA.MMRT', 'SG.GEN.PARL.ZS', 'SL.TLF.ACTI.FE.ZS', 'Short Name', 'Region', 'Income Group']

⚠️  MISSING VALUE SEBELUM IMPUTASI
  SP.ADO.TFRT: 0 missing (0.0%)
  SH.STA.MMRT: 23 missing (10.6%)
  SG.GEN.PARL.ZS: 24 missing (11.1%)
  SL.TLF.ACTI.FE.ZS: 30 missing (13.8%)


In [9]:
import pandas as pd
import numpy as np

# ============================================================
# STEP 6 — Imputasi Missing Value (Regional Median)
# ============================================================
# Convert ke Pandas (data sudah kecil: 217 baris)
df_pd = df_joined.toPandas()

# Imputasi: median regional -> median global (fallback)
for col_name in TARGET_INDICATORS:
    # Isi dengan median dari Region yang sama
    region_median = df_pd.groupby("Region")[col_name].transform("median")
    df_pd[col_name] = df_pd[col_name].fillna(region_median)
    # Fallback: jika masih null, isi dengan median global
    df_pd[col_name] = df_pd[col_name].fillna(df_pd[col_name].median())

print("=" * 50)
print("✅ MISSING VALUE SESUDAH IMPUTASI")
print("=" * 50)
print(df_pd[TARGET_INDICATORS].isnull().sum())
print("\n→ Semua harus bernilai 0!")

✅ MISSING VALUE SESUDAH IMPUTASI
SP.ADO.TFRT          0
SH.STA.MMRT          0
SG.GEN.PARL.ZS       0
SL.TLF.ACTI.FE.ZS    0
dtype: int64

→ Semua harus bernilai 0!


In [10]:
from sklearn.preprocessing import MinMaxScaler

# ============================================================
# STEP 7 — Normalisasi Min-Max (skala 0-1)
# ============================================================
scaler = MinMaxScaler()
X_normalized = scaler.fit_transform(df_pd[TARGET_INDICATORS])
df_normalized = pd.DataFrame(X_normalized, columns=TARGET_INDICATORS)

# Gabungkan kembali dengan metadata
df_silver = pd.concat([
    df_pd[["Country Code", "Country Name", "Region", "Income Group"]].reset_index(drop=True),
    df_normalized
], axis=1)

print("=" * 50)
print("✅ DATA SETELAH NORMALISASI MIN-MAX")
print("=" * 50)
print(f"Shape: {df_silver.shape}")
print(f"\nRentang data (harus 0-1):")
print(df_silver[TARGET_INDICATORS].describe().loc[['min', 'max']].to_string())
print(f"\nSample data:")
print(df_silver.head(5).to_string(index=False))

✅ DATA SETELAH NORMALISASI MIN-MAX
Shape: (212, 8)

Rentang data (harus 0-1):
     SP.ADO.TFRT  SH.STA.MMRT  SG.GEN.PARL.ZS  SL.TLF.ACTI.FE.ZS
min          0.0          0.0             0.0                0.0
max          1.0          1.0             1.0                1.0

Sample data:
Country Code Country Name                    Region        Income Group  SP.ADO.TFRT  SH.STA.MMRT  SG.GEN.PARL.ZS  SL.TLF.ACTI.FE.ZS
         GIB    Gibraltar     Europe & Central Asia         High income     0.050221     0.006348        0.438111           0.782615
         GTM    Guatemala Latin America & Caribbean Upper middle income     0.473893     0.097041        0.262979           0.441757
         ISL      Iceland     Europe & Central Asia         High income     0.028951     0.003401        0.679384           0.981084
         JPN        Japan       East Asia & Pacific         High income     0.014452     0.003174        0.157968           0.817186
         MWI       Malawi        Sub-Saharan Afr

In [11]:
# ============================================================
# SIMPAN KE SILVER LAYER
# ============================================================
df_silver_spark = spark.createDataFrame(df_silver)
df_silver_spark.write.mode("overwrite").parquet("/data/silver/sdg5_normalized.parquet")

print("=" * 50)
print("✅ SILVER LAYER SELESAI")
print("=" * 50)
print(f"Output: /data/silver/sdg5_normalized.parquet")
print(f"Total baris: {df_silver_spark.count()}")
print(f"Total kolom: {len(df_silver_spark.columns)}")
print(f"Kolom: {df_silver_spark.columns}")
print(f"Missing values: {df_silver[TARGET_INDICATORS].isnull().sum().sum()} (harus 0)")

✅ SILVER LAYER SELESAI
Output: /data/silver/sdg5_normalized.parquet
Total baris: 212
Total kolom: 8
Kolom: ['Country Code', 'Country Name', 'Region', 'Income Group', 'SP.ADO.TFRT', 'SH.STA.MMRT', 'SG.GEN.PARL.ZS', 'SL.TLF.ACTI.FE.ZS']
Missing values: 0 (harus 0)
